In [0]:
# =================================================================================
# Import necessary functions from Pyspark
# =================================================================================
from pyspark.sql.functions import col, regexp_replace, replace
from pyspark.sql.functions import first, round, date_diff
from pyspark.sql.functions import broadcast, when, lit
from pyspark.sql import functions as F
from pyspark.sql.functions import initcap
from pyspark.sql.functions import count, avg, sum
from pyspark.sql.functions import max, collect_set, countDistinct

In [0]:
# =================================================================================
# Function to map state codes to state names
# =================================================================================
def state_mapping():
    brazil_states = [
        ("AC", "Acre"), ("AL", "Alagoas"), ("AP", "Amapá"), ("AM", "Amazonas"),
        ("BA", "Bahia"), ("CE", "Ceará"), ("DF", "Distrito Federal"), ("ES", "Espírito Santo"),
        ("GO", "Goiás"), ("MA", "Maranhão"), ("MT", "Mato Grosso"), ("MS", "Mato Grosso do Sul"),
        ("MG", "Minas Gerais"), ("PA", "Pará"), ("PB", "Paraíba"), ("PR", "Paraná"),
        ("PE", "Pernambuco"), ("PI", "Piauí"), ("RJ", "Rio de Janeiro"), ("RN", "Rio Grande do Norte"),
        ("RS", "Rio Grande do Sul"), ("RO", "Rondônia"), ("RR", "Roraima"), ("SC", "Santa Catarina"),
        ("SP", "São Paulo"), ("SE", "Sergipe"), ("TO", "Tocantins")
    ]

    state_mapping_df = spark.createDataFrame(brazil_states, ["state_code", "state"])

    return state_mapping_df

In [0]:
# =================================================================================
# Read a table from ecommerce_brazil catalog
# =================================================================================
def read_table(table_name, layer, catalog="ecommerce_brazil"):
    return spark.sql(f"SELECT * FROM {catalog}.{layer}.{table_name}")

In [0]:
# =================================================================================
# Rename columns and use Camel Case
# =================================================================================
def clean_columns(df, rename_map = None, init_map = None, drop_cols = None):
    if rename_map:
        for old_col, new_col in rename_map.items():
            df = df.withColumnRenamed(old_col, new_col)
    if init_map:
        for col in init_map:
            df = df.withColumn(col, initcap(col))
    if drop_cols:
            df = df.drop(*drop_cols)
    return df

In [0]:
# =================================================================================
# Write tables into their respective schemas 
# =================================================================================
def write_table(df, table_name, schema_name, catalog="ecommerce_brazil", zorder_by=None):
    # Add metadata information to the table
    final_df = df.withColumn("_ingested_at", F.current_timestamp()) \
                    .withColumn("_source_file",  lit(table_name))
    
    # set the target path
    target_path = f"{catalog}.{schema_name}.{table_name}"
    print(f"Starting write for {schema_name}.{table_name}...")
    
   # 2. Configure the Writer
    writer = final_df.write \
        .format("delta") \
        .mode("overwrite") \
        .option("delta.autoOptimize.optimizeWrite", "true") \
        .option("delta.autoOptimize.autoCompact", "true")
    
    # Schema handling
    if schema_name == "silver":
        writer = writer.option("mergeSchema", "true")
    else:
        writer = writer.option("overwriteSchema", "true")
    
    # Execute the write operation
    writer.saveAsTable(target_path)
    print(f"Table {target_path} written successfully.")

    # Z-order the table for the given columns
    if zorder_by:
        print(f"Optimizing {target_path} with Z-Order: {zorder_by}")
        spark.sql(f"OPTIMIZE {target_path} ZORDER BY ({zorder_by})")

In [0]:
print(f"Common functions notebook has been read successfully")